In [19]:
import sys
print(sys.executable)

/root/disk2/runzhi/.conda/bin/python


In [20]:
!pip install anndata

In [21]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.io as sio
import harmonypy as hm
import anndata as ad



In [22]:

#filter expression matrices to only include HVGs shared across all datasets
def hvg_selection_and_pooling(exp_paths, n_top_genes = 1000):
    #input n expression matrices paths, output n expression matrices with only the union of the HVGs

    #read adata and find hvgs
    hvg_bools = []
    for d in exp_paths:
        adata = sio.mmread(d)
        adata = adata.toarray()
        print(adata.shape)
        adata = sc.AnnData(X=adata.T, dtype=adata.dtype)

        # Preprocess the data
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes)
        
        #save hvgs
        hvg = adata.var['highly_variable']
        hvg_bools.append(hvg)
    
    #find union of hvgs
    hvg_union = hvg_bools[0]
    for i in range(1, len(hvg_bools)):
        print(sum(hvg_union), sum(hvg_bools[i]))
        hvg_union = hvg_union | hvg_bools[i]

    print("Number of HVGs: ", hvg_union.sum())

    #filter expression matrices
    filtered_exp_mtxs = []
    for d in exp_paths:
        adata = sio.mmread(d)
        adata = adata.toarray()
        adata = sc.AnnData(X=adata.T, dtype=adata.dtype)

        # Preprocess the data and subset
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        filtered_exp_mtxs.append(adata[:, hvg_union].X)
    return filtered_exp_mtxs



exp_paths = [
    "GSE240429_data/data/filtered_expression_matrices/1/matrix.mtx",
    "GSE240429_data/data/filtered_expression_matrices/2/matrix.mtx",
    "GSE240429_data/data/filtered_expression_matrices/3/matrix.mtx",
    "GSE240429_data/data/filtered_expression_matrices/4/matrix.mtx",
]

filtered_mtx = hvg_selection_and_pooling(exp_paths)

for i in range(len(filtered_mtx)):
    np.save(
        "GSE240429_data/data/filtered_expression_matrices/" + str(i + 1) + "/hvg_matrix.npy",
        filtered_mtx[i].T,
    )

# import scanpy as sc
# import numpy as np
# import os

# def hvg_selection_and_pooling(exp_paths, n_top_genes=1000):
#     """
#     读取表达矩阵，计算每个数据集的 HVG，
#     并返回过滤至这些 HVG 并集的矩阵。
#     """
#     adatas = []
#     hvg_bools = []

#     # --- 第一阶段：读取、预处理并寻找 HVG ---
#     for d in exp_paths:
#         print(f"正在处理 {d}...")
        
#         # 1. 直接以稀疏矩阵形式读取并转置为 (细胞 x 基因)
#         # scanpy.read_mtx 比 scipy.io.mmread 更快且内存效率更高
#         adata = ad.read_mtx(d).T
#         print(f"  矩阵形状: {adata.shape}")

#         # 2. 预处理数据 (原生支持稀疏矩阵操作)
#         sc.pp.normalize_total(adata)
#         sc.pp.log1p(adata)

#         # 3. 寻找 HVG
#         sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes)
        
#         # 提取 HVG 的布尔数组 (.values 可防止 pandas 索引对齐问题)
#         hvg = adata.var['highly_variable'].values
#         hvg_bools.append(hvg)
        
#         # 将预处理后的 adata 保留在内存中，避免二次读取磁盘
#         adatas.append(adata)

#     # --- 第二阶段：寻找 HVG 的并集 ---
#     # np.logical_or.reduce 能够高效地跨多个数组应用按位或 (OR) 运算
#     hvg_union = np.logical_or.reduce(hvg_bools)
#     print(f"\n并集中唯一的 HVG 总数: {hvg_union.sum()}")

#     # --- 第三阶段：过滤表达矩阵 ---
#     filtered_exp_mtxs = []
#     for adata in adatas:
#         # 使用并集掩码 (mask) 进行子集提取
#         filtered_X = adata[:, hvg_union].X
        
#         # 仅在最后一步才转换为密集数组，以满足你原始输出的需求
#         if hasattr(filtered_X, "toarray"):
#             filtered_X = filtered_X.toarray()
            
#         filtered_exp_mtxs.append(filtered_X)

#     return filtered_exp_mtxs


# # --- 执行部分 ---
# exp_paths = [
#     "GSE240429_data/data/filtered_expression_matrices/1/matrix.mtx",
#     "GSE240429_data/data/filtered_expression_matrices/2/matrix.mtx",
#     "GSE240429_data/data/filtered_expression_matrices/3/matrix.mtx",
#     "GSE240429_data/data/filtered_expression_matrices/4/matrix.mtx",
# ]

# filtered_mtx = hvg_selection_and_pooling(exp_paths)

# # 根据输入目录动态保存输出结果
# for i, path in enumerate(exp_paths):
#     # 从路径中提取目录，例如 "GSE240429_data/data/filtered_expression_matrices/1"
#     dir_name = os.path.dirname(path) 
#     save_path = os.path.join(dir_name, "hvg_matrix.npy")
    
#     # 在保存之前转置回 (基因 x 细胞) 格式，与你原本的脚本保持一致
#     np.save(save_path, filtered_mtx[i].T)
#     print(f"已保存: {save_path}")

(36601, 2378)


/root/disk2/runzhi/.conda/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


(36601, 2349)


/root/disk2/runzhi/.conda/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


(36601, 2277)


/root/disk2/runzhi/.conda/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


(36601, 2265)


/root/disk2/runzhi/.conda/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


1000 1000
1888 1000
2703 1000
Number of HVGs:  3467


/root/disk2/runzhi/.conda/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/root/disk2/runzhi/.conda/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/root/disk2/runzhi/.conda/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/root/disk2/runzhi/.conda/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(


2026-03-29 13:21:58,871 - harmonypy - INFO - Running Harmony (PyTorch on cpu)
2026-03-29 13:21:58,872 - harmonypy - INFO -   Parameters:
2026-03-29 13:21:58,873 - harmonypy - INFO -     max_iter_harmony: 10
2026-03-29 13:21:58,874 - harmonypy - INFO -     max_iter_kmeans: 20
2026-03-29 13:21:58,875 - harmonypy - INFO -     epsilon_cluster: 1e-05
2026-03-29 13:21:58,878 - harmonypy - INFO -     epsilon_harmony: 0.0001
2026-03-29 13:21:58,879 - harmonypy - INFO -     nclust: 100
2026-03-29 13:21:58,880 - harmonypy - INFO -     block_size: 0.05
2026-03-29 13:21:58,881 - harmonypy - INFO -     lamb: [1. 1. 1. 1.]
2026-03-29 13:21:58,883 - harmonypy - INFO -     theta: [2. 2. 2. 2.]
2026-03-29 13:21:58,884 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-03-29 13:21:58,885 - harmonypy - INFO -     verbose: True
2026-03-29 13:21:58,885 - harmonypy - INFO -     random_state: 0
2026-03-29 13:21:58,886 - harmonypy - INFO -   Data: 3467 PCs × 9269 cells
2026-03-29 13:21:58,886 - har

(3467, 2378)
(3467, 2349)
(3467, 2277)
(3467, 2265)


2026-03-29 13:22:05,450 - harmonypy - INFO - KMeans initialization complete.
2026-03-29 13:22:05,462 - harmonypy - INFO - Iteration 1 of 10
2026-03-29 13:22:13,597 - harmonypy - INFO - Iteration 2 of 10
2026-03-29 13:22:22,205 - harmonypy - INFO - Iteration 3 of 10
2026-03-29 13:22:30,614 - harmonypy - INFO - Iteration 4 of 10
2026-03-29 13:22:38,855 - harmonypy - INFO - Converged after 4 iterations


(2378, 3467) (2349, 3467) (2277, 3467) (2265, 3467)


In [24]:
import os
import numpy as np
import pandas as pd

# 配置你的基础路径
base_dir = "/root/disk2/runzhi/BLEEP/GSE240429_data/data/filtered_expression_matrices"

print("-" * 60)
print(f"{'Slide':<10} | {'Barcodes (tsv)':<15} | {'Harmony (.npy)':<15} | {'HVG (.npy)':<15}")
print("-" * 60)

for slide in ["1", "2", "3", "4"]:
    slide_dir = os.path.join(base_dir, slide)
    
    # 1. 检查 barcodes 数量
    barcode_file = os.path.join(slide_dir, "barcodes.tsv")
    if os.path.exists(barcode_file):
        # 很多 barcode 文件没有 header，所以直接读取所有行
        with open(barcode_file, 'r') as f:
            num_barcodes = sum(1 for line in f if line.strip())
    else:
        num_barcodes = "Missing"

    # 2. 检查 harmony_matrix.npy 的细胞数 (行数)
    harmony_file = os.path.join(slide_dir, "harmony_matrix.npy")
    if os.path.exists(harmony_file):
        try:
            harmony_mat = np.load(harmony_file)
            num_harmony = harmony_mat.shape[0]
        except Exception:
            num_harmony = "Load Error"
    else:
        num_harmony = "Missing"

    # 3. 检查 hvg_matrix.npy 的细胞数 (行数)
    hvg_file = os.path.join(slide_dir, "hvg_matrix.npy")
    if os.path.exists(hvg_file):
        try:
            hvg_mat = np.load(hvg_file)
            num_hvg = hvg_mat.shape[0]
        except Exception:
            num_hvg = "Load Error"
    else:
        num_hvg = "Missing"

    print(f"Slide {slide:<4} | {str(num_barcodes):<15} | {str(num_harmony):<15} | {str(num_hvg):<15}")

print("-" * 60)

------------------------------------------------------------
Slide      | Barcodes (tsv)  | Harmony (.npy)  | HVG (.npy)     
------------------------------------------------------------
Slide 1    | 4992            | 3467            | 3467           
Slide 2    | 4992            | 3467            | 3467           
Slide 3    | 2277            | 3467            | 3467           
Slide 4    | 4992            | 3467            | 3467           
------------------------------------------------------------
